# Setup

In [1]:
%load_ext autoreload
%autoreload 2
from os import environ
from sys import path

from torch.backends import cudnn

# enforce more deterministic behavior
environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

path.append("..")

from processor.core.ir_system.lm_interface import LMInterface
from processor.model.interface.model_factory import get_embed_model, get_llm
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger

from logging import INFO
import os

/home/luthfi/miniconda3/envs/pneuma/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
USER_ID = "llm"
DATA_SOURCES = ["biomedical"]
INITIAL_PROMPT = """I’m interested in looking at tumor sample data from our recent studies—
could you show me what kinds of tumor types we have represented in the dataset? I want to get a sense of the distribution before narrowing down to anything specific."""

In [3]:
logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
llm_path = "model/weight/qwen3-8b"
embed_model_path = "model/weight/bge-base"
pneuma = LMInterface({
    "llm": get_llm(llm_path)(llm_path),
    "embed_model": get_embed_model()(embed_model_path),
}, logger)

In [4]:
from processor.core.ir_system.ir_data_model import (
    AbstractDocument,
    convert_retrieval_results_to_str,
)


def print_format_to_gpt(tables: list[AbstractDocument]):
    print(
        f"""SYSTEM OUTPUT:
```{convert_retrieval_results_to_str(tables)}```"""
    )

In [5]:
def print_initial_prompt_to_chatgpt(domain: str, question: str):
    print(f"""You are simulating a {domain} domain expert interacting with a basic table discovery system to explore insights from an enterprise dataset.

This system only supports static table retrieval based on your input description. It will return one or more tables that may be relevant to your query, but:
- The system does not infer your deeper intent.
- The system does not combine, transform, or analyze data for you.
- You must do all the reasoning yourself based on the returned tables.
          
Your task is to gradually explore and refine your question about some aspect of the data.
You do not begin with a precise question — your curiosity evolves step-by-step based on the tables you receive.
You must rely only on the table contents to guide your next query.

In this scenario:
- The system already has access to an internal {domain} dataset.
- You are familiar with the domain and have seen similar datasets before.
- You are not uploading new datasets or asking if data exists — you assume it does.
- The system does not explain its reasoning; it simply returns tables.

Here is a possible eventual goal (you do not know this at the start, and you may or may not reach it):

{question}

Your behavior should reflect:
- You are familiar with the domain but must infer relevant relationships from static tables.
- You refine your question slowly, sometimes going in the wrong direction.
- You may request more specific filters or different table fields in later turns.
- You will only reach the specific question above if you deduce it from the tables, which may take multiple turns.""")

print_initial_prompt_to_chatgpt(
    "biomedical", "What is the average age of patients with serous tumor samples analyzed in the study?"
)

You are simulating a biomedical domain expert interacting with a basic table discovery system to explore insights from an enterprise dataset.

This system only supports static table retrieval based on your input description. It will return one or more tables that may be relevant to your query, but:
- The system does not infer your deeper intent.
- The system does not combine, transform, or analyze data for you.
- You must do all the reasoning yourself based on the returned tables.

Your task is to gradually explore and refine your question about some aspect of the data.
You do not begin with a precise question — your curiosity evolves step-by-step based on the tables you receive.
You must rely only on the table contents to guide your next query.

In this scenario:
- The system already has access to an internal biomedical dataset.
- You are familiar with the domain and have seen similar datasets before.
- You are not uploading new datasets or asking if data exists — you assume it does.


# INTERACTION

In [6]:
tables = pneuma.retrieve_documents(
    INITIAL_PROMPT,
    DATA_SOURCES,
    10,
    [RetrieverType.PNEUMA],
)[RetrieverType.PNEUMA]
print_format_to_gpt(tables)

[2025-07-29 19:17:46] INFO in lm_interface: Starting document retrieval for prompt: I’m interested in looking at tumor sample data from our recent studies—
could you show me what kinds...
[2025-07-29 19:17:46] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>]
[2025-07-29 19:17:46] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


[2025-07-29 19:17:47] INFO in lm_interface: => Initial retrieval returned 1 documents:
[2025-07-29 19:17:47] INFO in lm_interface: ==> Table ../../data_src/biomedical/dataset/UCEC_CPTAC3_meta_table_V2.1:
col: idx | Proteomics_Participant_ID | Case_excluded | Proteomics_TMT_batch | Proteomics_TMT_plex | Proteomics_TMT_channel | Proteomics_Parent_Sample_IDs | Proteomics_Aliquot_ID | Proteomics_Tumor_Normal | Proteomics_OCT | Country | Histologic_Grade_FIGO | Myometrial_invasion_Specify | Histologic_type | Treatment_naive | Tumor_purity | Path_Stage_Primary_Tumor-pT | Path_Stage_Reg_Lymph_Nodes-pN | Clin_Stage_Dist_Mets-cM | Path_Stage_Dist_Mets-pM | tumor_Stage-Pathological | FIGO_stage | LVSI | BMI | Age | Diabetes | Race | Ethnicity | Gender | Tumor_Site | Tumor_Site_Other | Tumor_Focality | Tumor_Size_cm | Estrogen_Receptor | Estrogen_Receptor_% | Progesterone_Receptor | Progesterone_Receptor_% | MLH1 | MLH2 | MSH6 | PMS2 | p53 | Other_IHC_specify | MLH1_Promoter_Hypermethylation | Nu

In [7]:
tables = pneuma.retrieve_documents(
    """Alright — now that I have a sense of the tumor types available in the dataset, could you show me a simple frequency table of Histologic_type so I can see how many samples fall into each category?

That way I can spot if there are particular subtypes worth focusing on for further exploration.""",
    DATA_SOURCES,
    10,
    [RetrieverType.PNEUMA],
)[RetrieverType.PNEUMA]
print_format_to_gpt(tables)

[2025-07-29 19:20:47] INFO in lm_interface: Starting document retrieval for prompt: Alright — now that I have a sense of the tumor types available in the dataset, could you show me a s...
[2025-07-29 19:20:47] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>]
[2025-07-29 19:20:47] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


[2025-07-29 19:20:47] INFO in lm_interface: => Initial retrieval returned 3 documents:
[2025-07-29 19:20:47] INFO in lm_interface: ==> Table ../../data_src/biomedical/dataset/UCEC_CPTAC3_meta_table_V2.1:
col: idx | Proteomics_Participant_ID | Case_excluded | Proteomics_TMT_batch | Proteomics_TMT_plex | Proteomics_TMT_channel | Proteomics_Parent_Sample_IDs | Proteomics_Aliquot_ID | Proteomics_Tumor_Normal | Proteomics_OCT | Country | Histologic_Grade_FIGO | Myometrial_invasion_Specify | Histologic_type | Treatment_naive | Tumor_purity | Path_Stage_Primary_Tumor-pT | Path_Stage_Reg_Lymph_Nodes-pN | Clin_Stage_Dist_Mets-cM | Path_Stage_Dist_Mets-pM | tumor_Stage-Pathological | FIGO_stage | LVSI | BMI | Age | Diabetes | Race | Ethnicity | Gender | Tumor_Site | Tumor_Site_Other | Tumor_Focality | Tumor_Size_cm | Estrogen_Receptor | Estrogen_Receptor_% | Progesterone_Receptor | Progesterone_Receptor_% | MLH1 | MLH2 | MSH6 | PMS2 | p53 | Other_IHC_specify | MLH1_Promoter_Hypermethylation | Nu